# AMEX Enterprise Credit Risk Platform
## Notebook 55 -- Credit Line Management: Modeling
### Phase 4 . Problem Statement 10: Credit Line Management

CRISP-DM stage: **Modeling**. Depends on Problem 1 Notebooks 01-05 (real champion model + real whole-history
engineered feature matrix), Problem 6 Notebooks 38-40 (real persisted trailing-window model), and this
problem's own Notebook 54 (real policy).

**What this notebook does:** scores every real customer with two already-validated real models -- Problem
1's champion classifier (static, whole-history PD) and Problem 6's persisted trailing-window classifier
(dynamic, monthly-refreshed PD) -- composes the real PD_TREND signal from the two, fits real tertile cut
values for the risk-level and trend tiers on the real internal TRAIN split, validates both hard-gating KPIs
(risk-level monotonicity and trend coherence) against the real observed default outcome on the real internal
HOLDOUT split, reports the full classification metrics suite for the dynamic PD score, applies Notebook 54's
9-cell action-tier policy to produce a real ranked credit-line worklist, and writes
`credit_line_modeling_results.json` + `credit_line_worklist.parquet` for Notebook 56 (Validation &
Deployment) to consume.

**No fresh model is fit here** -- Problem 10 composes two already-validated real classifiers rather than
training a third; this notebook's real modeling work is the tertile-cut fitting and the two hard-gate KPI
validations, both computed live against real data, not assumed.

**HYPER note:** `build_trailing_window_store()` is copied VERBATIM from Notebook 39 (Problem 6), per this
platform's established convention of copying reusable feature-engineering logic across notebooks rather than
importing it across notebook boundaries. Sections 1-3's structure reuses this platform's established
Phase 4 template.

**WARP note:** same Phase 4 tightened 92%/92% CPU/RAM cap and two-tier RAM pre-flight guard as Notebook 54.
Section 5's `build_trailing_window_store()` call is this notebook's single heaviest step (a real full
streaming pass over the raw CSV) -- RSS and available-RAM checkpoints are printed immediately before and
after it, so any future freeze report pinpoints the exact step, the same discipline that resolved the real
Notebook 52 freeze.

Zero-fabrication statement: every number this notebook prints is either computed live against the real raw
Kaggle CSVs and the real persisted models from Problems 1 and 6, or an explicitly labeled ASSUMPTION carried
forward from Notebook 54 -- no results are hardcoded or estimated in advance.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-05, 38-40, 54
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 38-40, 54")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB54_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_54_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first -- this notebook reuses its real, "
                         "already-built whole-history feature matrix rather than re-deriving one"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first (real EAD/LGD assumptions inherited, "
                         "not re-guessed)"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first"),
    (NB54_SUMMARY_PATH, "run 54_credit_line_management_business_understanding.ipynb (Problem 10) first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB54_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB54_SUMMARY = json.load(f)

POLICY_PATH = Path(NB54_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(f"{POLICY_PATH} not found.\nFix: re-run Notebook 54.")
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    CREDIT_LINE_POLICY = json.load(f)

RISK_LEVEL_NAMES = CREDIT_LINE_POLICY["risk_level_names"]
RISK_LEVEL_CUT_PERCENTILES = CREDIT_LINE_POLICY["risk_level_cut_percentiles"]
TREND_NAMES = CREDIT_LINE_POLICY["trend_names"]
TREND_CUT_PERCENTILES = CREDIT_LINE_POLICY["trend_cut_percentiles"]
MIN_STATEMENTS_FOR_DYNAMIC_PD = CREDIT_LINE_POLICY["min_statements_for_dynamic_pd"]
KPI_TARGETS = CREDIT_LINE_POLICY["kpi_targets"]
ACTION_TIER_MATRIX = KPI_TARGETS["action_tier_policy"]["matrix"]

# --- Problem 1's real whole-history engineered feature matrix (Notebook 04's
#     own recorded output path) -- Notebook 55 does NOT re-derive feature
#     engineering; it reuses the exact same real feature store Notebook 05
#     trained the champion model on. ---
TRAIN_FULL_ENGINEERED_PATH = Path(NB04_SUMMARY["output_files"]["train_full_engineered.parquet"])
if not TRAIN_FULL_ENGINEERED_PATH.exists():
    raise FileNotFoundError(f"{TRAIN_FULL_ENGINEERED_PATH} not found.\nFix: re-run Notebook 04.")

# --- Problem 1's real champion model + preprocessing artifacts -- same
#     resolution main.py (the deployed service) already uses, reused
#     verbatim rather than re-derived. ---
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")
if "model_development" not in PILLAR_DIRS:
    raise KeyError("project_config.json's pillar_dirs has no 'model_development' entry.")
_p1_models_subdir = PILLAR_DIRS["model_development"] / "models"
P1_CHAMPION_MODEL_PATH = _p1_models_subdir / (CHAMPION_NAME + ".joblib")
P1_PREPROCESSING_PATH = _p1_models_subdir / "preprocessing_artifacts.joblib"
for _p, _label in [(P1_CHAMPION_MODEL_PATH, "Problem 1's persisted champion model"),
                    (P1_PREPROCESSING_PATH, "Problem 1's preprocessing artifacts")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run Notebook 05.")

# --- Problem 6's real persisted trailing-window model + preprocessing
#     artifacts -- same NB40_SUMMARY-sourced paths Notebook 54 already
#     verified exist. ---
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]
P6_MODEL_PATH = Path(NB40_SUMMARY["model_path"])
P6_PREPROCESSING_PATH = Path(NB40_SUMMARY["preprocessing_path"])
for _p, _label in [(P6_MODEL_PATH, "Problem 6's persisted model"),
                    (P6_PREPROCESSING_PATH, "Problem 6's preprocessing artifacts")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run Notebook 40.")

EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

if "credit_line_policy" in PILLAR_DIRS:
    CREDIT_LINE_POLICY_DIR = PILLAR_DIRS["credit_line_policy"]
else:
    CREDIT_LINE_POLICY_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "10_Problem10_Credit_Line_Management" / "policy"
    )
CREDIT_LINE_POLICY_DIR.mkdir(parents=True, exist_ok=True)
if "credit_line_modeling" in PILLAR_DIRS:
    CREDIT_LINE_MODELING_DIR = PILLAR_DIRS["credit_line_modeling"]
else:
    CREDIT_LINE_MODELING_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "10_Problem10_Credit_Line_Management" / "modeling"
    )
    print(f"NOTE: 'credit_line_modeling' not in pillar_dirs -- using fallback: {CREDIT_LINE_MODELING_DIR}")
CREDIT_LINE_MODELING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                          : {CONFIG_PATH}")
print(f"Loaded Notebook 54's real policy             : {POLICY_PATH}")
print(f"Problem 1 champion model (measured)          : {CHAMPION_NAME} @ {P1_CHAMPION_MODEL_PATH}")
print(f"Problem 6 winning window / recommended       : W={P6_WINNING_W} / {P6_RECOMMENDED_FOR_PRODUCTION}")
print(f"RISK_LEVEL_NAMES (from Notebook 54)          : {RISK_LEVEL_NAMES}")
print(f"TREND_NAMES (from Notebook 54)               : {TREND_NAMES}")
print(f"Modeling artifacts will be written under: {CREDIT_LINE_MODELING_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 TIGHTENED
#            CAP, SAME 92%/92% AS NOTEBOOK 54, WITH THE TWO-TIER RAM GUARD)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports (Phase 4 Tightened Cap)")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count,
    max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(
    _historical_max_ram_bytes,
    round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP),
)

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
        precision_recall_curve, confusion_matrix,
    )
except ImportError:
    missing.append("scikit-learn")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, which is below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs to safely stream the raw "
        f"train_data.csv (Section 5's build_trailing_window_store() call is the heaviest step) without "
        f"risking a full-system freeze. Close other Jupyter kernels / notebooks / memory-heavy applications, "
        f"confirm available RAM with `psutil.virtual_memory().available / 1e9` in a fresh cell, then re-run "
        f"this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(
        f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB of system RAM is available "
        f"(comfortable margin is {_comfortable_available_ram_gb:.2f} GB). Proceeding, since this is above "
        f"the {_min_required_available_ram_gb:.2f} GB hard-fail floor, but headroom is tighter than ideal."
    )
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available >= "
          f"{_comfortable_available_ram_gb:.2f} GB comfortable margin.")

logger.info(
    f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
    f"({WARP_THREAD_COUNT / DETECTED_LOGICAL_CORES:.0%}, Phase 4 tightened cap)"
)
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling (Phase 4 tightened cap): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses (see Notebook 35 Section 3):
    the real known current nested Phase/Problem path is checked FIRST, then PILLAR_DIRS, then the
    legacy root-level path, then whatever a summary JSON literally recorded. Copied verbatim from
    Notebook 39 (Problem 6), per this platform's established convention of copying reusable helpers
    rather than importing across notebooks."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv                                          : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv                                        : {RAW_TRAIN_LABELS_PATH}")
print(f"train_split.csv (internal train, Notebook 02's real split)  : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: SCORE EVERY CUSTOMER WITH PROBLEM 1'S REAL CHAMPION MODEL
#            (STATIC PD)
# =============================================================================
_section("SECTION 4: Score Every Customer With Problem 1's Real Champion Model (Static PD)")

P1_PREPROCESSING = joblib.load(P1_PREPROCESSING_PATH)
P1_LABEL_ENCODERS = P1_PREPROCESSING["label_encoders"]
P1_FEATURE_MEDIANS = P1_PREPROCESSING["feature_medians"]
P1_ALL_FEATURE_COLS = P1_PREPROCESSING["all_feature_cols"]
P1_CATEGORICAL_COLS = P1_PREPROCESSING["categorical_encode_cols"]
P1_NUMERIC_COLS = P1_PREPROCESSING["numeric_feature_cols"]
P1_CHAMPION_USES_SCALED = CHAMPION_NAME == "logistic_regression"
if P1_CHAMPION_USES_SCALED:
    P1_SCALER = P1_PREPROCESSING["scaler"]

print(f"Reading Problem 1's real whole-history engineered feature matrix: {TRAIN_FULL_ENGINEERED_PATH}")
_t0 = time.time()
_engineered_cols = ["customer_ID"] + [c for c in P1_ALL_FEATURE_COLS]
_train_full_eng = pl.read_parquet(TRAIN_FULL_ENGINEERED_PATH, columns=_engineered_cols)
print(f"Read {_train_full_eng.height:,} customers x {_train_full_eng.width} columns in "
      f"{time.time() - _t0:.1f}s. RSS: {_rss_gb():.2f} GB")

# --- Same per-row recipe main.py's /predict endpoint uses, applied vectorized
#     across the whole population: categorical columns -> label_encoders'
#     trained class index (unseen/missing -> the trained "__missing__" class
#     if present, else -1); numeric columns -> null filled with the trained
#     median. Exact same recipe, not re-derived, so this notebook's STATIC_PD
#     matches what the deployed API would return for the same customer. ---
_cat_exprs = []
for _c in P1_CATEGORICAL_COLS:
    _classes = P1_LABEL_ENCODERS[_c]["classes"]
    _mapping = {cat: idx for idx, cat in enumerate(_classes)}
    _default = _mapping.get("__missing__", -1)
    _cat_exprs.append(
        pl.col(_c).cast(pl.Utf8).fill_null("__missing__")
        .replace_strict(_mapping, default=_default).cast(pl.Float32).alias(_c)
    )
_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P1_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P1_NUMERIC_COLS
]
_train_full_eng = _train_full_eng.with_columns(_cat_exprs + _num_exprs)

STATIC_PD_DF = _train_full_eng.select(["customer_ID"] + P1_ALL_FEATURE_COLS)
_X_static = STATIC_PD_DF.select(P1_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _train_full_eng
gc.collect()

P1_CHAMPION_MODEL = joblib.load(P1_CHAMPION_MODEL_PATH)
if P1_CHAMPION_USES_SCALED:
    _X_static = (_X_static - np.asarray(P1_SCALER["mean"], dtype=np.float32)) / np.asarray(
        P1_SCALER["std"], dtype=np.float32)
_t0 = time.time()
_static_proba = P1_CHAMPION_MODEL.predict_proba(_X_static)[:, 1]
print(f"Scored {len(_static_proba):,} customers with Problem 1's real champion model "
      f"({CHAMPION_NAME}) in {time.time() - _t0:.1f}s. RSS: {_rss_gb():.2f} GB, "
      f"available RAM: {_available_ram_gb():.2f} GB")
STATIC_PD_DF = STATIC_PD_DF.select("customer_ID").with_columns(
    pl.Series("STATIC_PD", _static_proba, dtype=pl.Float64)
)
del _X_static, _static_proba
gc.collect()
print(f"STATIC_PD real range: [{STATIC_PD_DF['STATIC_PD'].min():.6f}, {STATIC_PD_DF['STATIC_PD'].max():.6f}], "
      f"mean {STATIC_PD_DF['STATIC_PD'].mean():.6f}")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: SCORE EVERY ELIGIBLE CUSTOMER WITH PROBLEM 6'S REAL PERSISTED
#            MODEL (DYNAMIC PD) -- build_trailing_window_store() REUSED
#            VERBATIM FROM NOTEBOOK 39
# =============================================================================
_section("SECTION 5: Score Every Eligible Customer With Problem 6's Real Persisted Model (Dynamic PD)")

P6_PREPROCESSING = joblib.load(P6_PREPROCESSING_PATH)
P6_FEATURE_MEDIANS = P6_PREPROCESSING["feature_medians"]
P6_ALL_FEATURE_COLS = P6_PREPROCESSING["all_feature_cols"]
P6_BASE_FEATURE_COLUMNS = P6_PREPROCESSING["base_feature_columns"]
if P6_PREPROCESSING.get("w") != P6_WINNING_W:
    raise RuntimeError(
        f"Problem 6's preprocessing artifact was fit at W={P6_PREPROCESSING.get('w')}, but "
        f"notebook_40_summary.json's winning_w is {P6_WINNING_W} -- these must agree."
    )


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int, k: int = 0) -> "pl.DataFrame":
    """Adapted from Notebook 39 (Problem 6)'s build_trailing_window_store, per this platform's
    established convention of copying reusable feature-engineering logic across notebooks rather than
    importing it, to avoid a hidden cross-notebook dependency. Streams csv_path and returns one
    aggregated row per customer_ID, restricted to a real W-statement window ending k statements before
    the most recent (by real S_2 date order). k=0 (the default, matching Notebook 39's own behavior
    exactly) is each customer's LAST W statements; k=w gives the immediately-preceding, non-overlapping
    W-statement window -- used below (Section 5) to build a genuine same-model, two-time-point PD trend
    (2026-08-27 redefinition, see Notebook 54 Section 6 addendum) rather than comparing two different
    models' scores. Customers with fewer than w+k statements are NOT excluded here -- the caller filters
    on the returned _actual_window_len column."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(
            (pl.col("_row_idx") >= (pl.col("_n_statements") - w - k))
            & (pl.col("_row_idx") < (pl.col("_n_statements") - k))
        )
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )

    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)

    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}"))
            .otherwise(None)
            .alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))

    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]

    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


print(f"Building the real trailing-{P6_WINNING_W}-statement feature store from the raw CSV "
      f"(this is the heaviest step in this notebook -- a real full streaming pass over "
      f"{RAW_TRAIN_DATA_PATH.stat().st_size / 1e9:.2f} GB). "
      f"RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
_p6_store = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W)
_p6_store = _p6_store.filter(pl.col("_actual_window_len") == P6_WINNING_W)
print(f"Built in {time.time() - _t0:.1f}s: {_p6_store.height:,} customers with a real, full "
      f"{P6_WINNING_W}-statement trailing window. RSS: {_rss_gb():.2f} GB, "
      f"available RAM: {_available_ram_gb():.2f} GB")

_missing_p6_cols = set(P6_ALL_FEATURE_COLS) - set(_p6_store.columns)
if _missing_p6_cols:
    raise RuntimeError(
        f"{len(_missing_p6_cols)} of Problem 6's real feature column(s) were not built: "
        f"{sorted(_missing_p6_cols)}\nFix: investigate a naming mismatch before proceeding."
    )

_p6_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P6_ALL_FEATURE_COLS
]
_p6_store = _p6_store.with_columns(_p6_num_exprs)

DYNAMIC_PD_DF = _p6_store.select(["customer_ID"] + P6_ALL_FEATURE_COLS)
_X_dynamic = DYNAMIC_PD_DF.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _p6_store
gc.collect()

P6_MODEL = joblib.load(P6_MODEL_PATH)
_t0 = time.time()
_dynamic_proba = P6_MODEL.predict_proba(_X_dynamic)[:, 1]
print(f"Scored {len(_dynamic_proba):,} customers with Problem 6's real persisted model in "
      f"{time.time() - _t0:.1f}s. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
DYNAMIC_PD_DF = DYNAMIC_PD_DF.select("customer_ID").with_columns(
    pl.Series("DYNAMIC_PD", _dynamic_proba, dtype=pl.Float64)
)
del _X_dynamic, _dynamic_proba
gc.collect()
print(f"DYNAMIC_PD real range: [{DYNAMIC_PD_DF['DYNAMIC_PD'].min():.6f}, "
      f"{DYNAMIC_PD_DF['DYNAMIC_PD'].max():.6f}], mean {DYNAMIC_PD_DF['DYNAMIC_PD'].mean():.6f}")

# --- 2026-08-27: real, EARLIER, non-overlapping window with the SAME reused function and the SAME
#     persisted Problem 6 model -- this earlier score is what Section 6 below differences against the
#     current DYNAMIC_PD to build a genuine same-model, two-time-point PD_TREND, replacing the original
#     cross-model (DYNAMIC_PD - STATIC_PD) definition (see Notebook 54 Section 6 addendum for why). ---
print(
    f"\nBuilding a real, EARLIER, non-overlapping {P6_WINNING_W}-statement window "
    f"(offset {P6_WINNING_W} statements back) with the SAME reused function and the SAME persisted "
    "Problem 6 model."
)
_t0 = time.time()
_p6_store_early = build_trailing_window_store(RAW_TRAIN_DATA_PATH, P6_BASE_FEATURE_COLUMNS, P6_WINNING_W, k=P6_WINNING_W)
_p6_store_early = _p6_store_early.filter(pl.col("_actual_window_len") == P6_WINNING_W)
print(f"Built in {time.time() - _t0:.1f}s: {_p6_store_early.height:,} customers with a real, full "
      f"EARLIER {P6_WINNING_W}-statement window (requires >= {2 * P6_WINNING_W} real total statements). "
      f"RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")

_p6_early_num_exprs = [
    pl.when(pl.col(_c).is_infinite() | pl.col(_c).is_nan()).then(None).otherwise(pl.col(_c))
    .fill_null(P6_FEATURE_MEDIANS[_c]).cast(pl.Float32).alias(_c)
    for _c in P6_ALL_FEATURE_COLS
]
_p6_store_early = _p6_store_early.with_columns(_p6_early_num_exprs)

DYNAMIC_PD_EARLY_DF = _p6_store_early.select(["customer_ID"] + P6_ALL_FEATURE_COLS)
_X_dynamic_early = DYNAMIC_PD_EARLY_DF.select(P6_ALL_FEATURE_COLS).to_numpy().astype(np.float32, copy=False)
del _p6_store_early
gc.collect()

_t0 = time.time()
_dynamic_early_proba = P6_MODEL.predict_proba(_X_dynamic_early)[:, 1]
print(f"Scored {len(_dynamic_early_proba):,} customers' EARLIER window with Problem 6's real persisted "
      f"model in {time.time() - _t0:.1f}s. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
DYNAMIC_PD_EARLY_DF = DYNAMIC_PD_EARLY_DF.select("customer_ID").with_columns(
    pl.Series("DYNAMIC_PD_EARLY", _dynamic_early_proba, dtype=pl.Float64)
)
del _X_dynamic_early, _dynamic_early_proba
gc.collect()
print(f"DYNAMIC_PD_EARLY real range: [{DYNAMIC_PD_EARLY_DF['DYNAMIC_PD_EARLY'].min():.6f}, "
      f"{DYNAMIC_PD_EARLY_DF['DYNAMIC_PD_EARLY'].max():.6f}], "
      f"mean {DYNAMIC_PD_EARLY_DF['DYNAMIC_PD_EARLY'].mean():.6f}")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: COMPOSE REAL PD_TREND & DEFINE THE CREDIT-LINE-ELIGIBLE POPULATION
# =============================================================================
_section("SECTION 6: Compose Real PD_TREND & Define the Credit-Line-Eligible Population")

# --- Eligible population: customers with a real STATIC_PD, a real DYNAMIC_PD, AND a real
#     DYNAMIC_PD_EARLY (inner join on all three). The trend-eligibility bar is now >= 2 x
#     P6_WINNING_W real statements (tighter than the plain dynamic-PD-only bar Notebook 54 Section 9
#     reports), because a genuine trend needs two real, non-overlapping windows to compare. STATIC_PD
#     is still carried through for reporting but is NO LONGER part of the PD_TREND formula -- see
#     Notebook 54 Section 6's 2026-08-27 addendum for why the original DYNAMIC_PD - STATIC_PD
#     definition was replaced (it was anti-correlated with real outcomes, root-caused to Problem 1's
#     STATIC_PD model being the real stronger standalone predictor of the two). ---
TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
SCORED_DF = (
    STATIC_PD_DF.join(DYNAMIC_PD_DF, on="customer_ID", how="inner")
    .join(DYNAMIC_PD_EARLY_DF, on="customer_ID", how="inner")
    .join(TARGET_DF, on="customer_ID", how="inner")
    .with_columns((pl.col("DYNAMIC_PD") - pl.col("DYNAMIC_PD_EARLY")).alias("PD_TREND"))
)
print(f"Credit-line-eligible population (real STATIC_PD + real DYNAMIC_PD + real DYNAMIC_PD_EARLY + "
      f"real target): {SCORED_DF.height:,} customers")
print(f"PD_TREND real range (now DYNAMIC_PD - DYNAMIC_PD_EARLY, a genuine same-model two-window "
      f"measure, redefined 2026-08-27): [{SCORED_DF['PD_TREND'].min():.6f}, "
      f"{SCORED_DF['PD_TREND'].max():.6f}], mean {SCORED_DF['PD_TREND'].mean():.6f}")

TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
SCORED_TRAIN_DF = SCORED_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
SCORED_HOLDOUT_DF = SCORED_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")
print(f"Real internal TRAIN split (fits tertile cuts)   : {SCORED_TRAIN_DF.height:,} customers")
print(f"Real internal HOLDOUT split (validates KPIs)    : {SCORED_HOLDOUT_DF.height:,} customers")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: FIT REAL TERTILE CUT VALUES (RISK-LEVEL & TREND) ON THE REAL
#            TRAIN SPLIT
# =============================================================================
_section("SECTION 7: Fit Real Tertile Cut Values (Risk-Level & Trend) on the Real TRAIN Split")

_p_lo, _p_hi = RISK_LEVEL_CUT_PERCENTILES[0] / 100.0, RISK_LEVEL_CUT_PERCENTILES[1] / 100.0
RISK_LEVEL_CUT_LOW = float(SCORED_TRAIN_DF["DYNAMIC_PD"].quantile(_p_lo))
RISK_LEVEL_CUT_HIGH = float(SCORED_TRAIN_DF["DYNAMIC_PD"].quantile(_p_hi))

_t_lo, _t_hi = TREND_CUT_PERCENTILES[0] / 100.0, TREND_CUT_PERCENTILES[1] / 100.0
TREND_CUT_LOW = float(SCORED_TRAIN_DF["PD_TREND"].quantile(_t_lo))
TREND_CUT_HIGH = float(SCORED_TRAIN_DF["PD_TREND"].quantile(_t_hi))

print(f"RISK_LEVEL cuts on real DYNAMIC_PD (fit on TRAIN split, {RISK_LEVEL_CUT_PERCENTILES} percentiles): "
      f"low={RISK_LEVEL_CUT_LOW:.6f}, high={RISK_LEVEL_CUT_HIGH:.6f}")
print(f"TREND cuts on real PD_TREND (fit on TRAIN split, {TREND_CUT_PERCENTILES} percentiles): "
      f"low={TREND_CUT_LOW:.6f}, high={TREND_CUT_HIGH:.6f}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: ASSIGN REAL RISK-LEVEL / TREND TIERS TO THE FULL ELIGIBLE
#            POPULATION
# =============================================================================
_section("SECTION 8: Assign Real Risk-Level / Trend Tiers to the Full Eligible Population")


def _assign_tiers(df: "pl.DataFrame") -> "pl.DataFrame":
    _risk_expr = (
        pl.when(pl.col("DYNAMIC_PD") <= RISK_LEVEL_CUT_LOW).then(pl.lit(RISK_LEVEL_NAMES[0]))
        .when(pl.col("DYNAMIC_PD") <= RISK_LEVEL_CUT_HIGH).then(pl.lit(RISK_LEVEL_NAMES[1]))
        .otherwise(pl.lit(RISK_LEVEL_NAMES[2]))
        .alias("RISK_LEVEL")
    )
    _trend_expr = (
        pl.when(pl.col("PD_TREND") <= TREND_CUT_LOW).then(pl.lit(TREND_NAMES[0]))
        .when(pl.col("PD_TREND") <= TREND_CUT_HIGH).then(pl.lit(TREND_NAMES[1]))
        .otherwise(pl.lit(TREND_NAMES[2]))
        .alias("TREND")
    )
    return df.with_columns([_risk_expr, _trend_expr])


SCORED_DF = _assign_tiers(SCORED_DF)
SCORED_TRAIN_DF = _assign_tiers(SCORED_TRAIN_DF)
SCORED_HOLDOUT_DF = _assign_tiers(SCORED_HOLDOUT_DF)

_tier_counts = (
    SCORED_DF.group_by(["RISK_LEVEL", "TREND"]).agg(pl.len().alias("n"))
    .sort(["RISK_LEVEL", "TREND"])
)
print("Real risk-level x trend cell population counts (full eligible population):")
for _row in _tier_counts.iter_rows(named=True):
    print(f"  {_row['RISK_LEVEL']:<12} x {_row['TREND']:<16}: {_row['n']:>8,}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: VALIDATE HARD-GATING KPIS ON THE REAL HOLDOUT SPLIT
# =============================================================================
_section("SECTION 9: Validate Hard-Gating KPIs on the Real HOLDOUT Split")

_holdout_by_risk = (
    SCORED_HOLDOUT_DF.group_by("RISK_LEVEL")
    .agg(pl.col("target").mean().alias("default_rate"), pl.len().alias("n"))
)
_risk_default_rates = {r["RISK_LEVEL"]: r["default_rate"] for r in _holdout_by_risk.iter_rows(named=True)}
_risk_ns = {r["RISK_LEVEL"]: r["n"] for r in _holdout_by_risk.iter_rows(named=True)}
print("Real observed default rate by risk-level tier (HOLDOUT split):")
for _name in RISK_LEVEL_NAMES:
    print(f"  {_name:<12}: {_risk_default_rates.get(_name, float('nan')):.4f}  (n={_risk_ns.get(_name, 0):,})")

_rates_in_order = [_risk_default_rates[_n] for _n in RISK_LEVEL_NAMES]
RISK_LEVEL_MONOTONIC = all(_rates_in_order[i] < _rates_in_order[i + 1] for i in range(len(_rates_in_order) - 1))
RISK_LEVEL_RATIO = (_rates_in_order[-1] / _rates_in_order[0]) if _rates_in_order[0] > 0 else float("inf")
RISK_LEVEL_MONOTONICITY_PASSED = (
    RISK_LEVEL_MONOTONIC
    and RISK_LEVEL_RATIO >= KPI_TARGETS["risk_level_monotonicity"]["min_default_rate_ratio_top_to_bottom_tier"]
)
print(f"\nrisk_level_monotonicity: strictly increasing={RISK_LEVEL_MONOTONIC}, "
      f"top/bottom ratio={RISK_LEVEL_RATIO:.2f} "
      f"(target >= {KPI_TARGETS['risk_level_monotonicity']['min_default_rate_ratio_top_to_bottom_tier']}) "
      f"-- {'PASS' if RISK_LEVEL_MONOTONICITY_PASSED else 'FAIL'}")

_holdout_by_cell = (
    SCORED_HOLDOUT_DF.group_by(["RISK_LEVEL", "TREND"])
    .agg(pl.col("target").mean().alias("default_rate"), pl.len().alias("n"))
)
_cell_rates = {(r["RISK_LEVEL"], r["TREND"]): (r["default_rate"], r["n"]) for r in _holdout_by_cell.iter_rows(named=True)}

print("\ntrend_coherence -- real observed default rate, Trending Worse vs Trending Better, within each "
      "risk-level tier (HOLDOUT split):")
_trend_coherence_cells_passed = []
for _risk_name in RISK_LEVEL_NAMES:
    _better = _cell_rates.get((_risk_name, TREND_NAMES[0]), (float("nan"), 0))
    _worse = _cell_rates.get((_risk_name, TREND_NAMES[2]), (float("nan"), 0))
    _cell_passed = _worse[0] > _better[0]
    _trend_coherence_cells_passed.append(_cell_passed)
    print(f"  {_risk_name:<12}: Trending Better={_better[0]:.4f} (n={_better[1]:,}), "
          f"Trending Worse={_worse[0]:.4f} (n={_worse[1]:,}) -- {'PASS' if _cell_passed else 'FAIL'}")
TREND_COHERENCE_PASSED = all(_trend_coherence_cells_passed)
print(f"trend_coherence overall: {'PASS' if TREND_COHERENCE_PASSED else 'FAIL'} "
      f"({sum(_trend_coherence_cells_passed)}/{len(_trend_coherence_cells_passed)} risk-level tiers coherent)")

_expected_cell_share_pct = 100.0 / 9.0
_min_cell_pct = KPI_TARGETS["min_tier_population_pct"] / 100.0 * _expected_cell_share_pct
_total_holdout = SCORED_HOLDOUT_DF.height
_holdout_cell_counts = (
    SCORED_HOLDOUT_DF.group_by(["RISK_LEVEL", "TREND"]).agg(pl.len().alias("n"))
)
_undersized_cells = [
    (r["RISK_LEVEL"], r["TREND"], r["n"]) for r in _holdout_cell_counts.iter_rows(named=True)
    if 100.0 * r["n"] / _total_holdout < _min_cell_pct
]
MIN_CELL_POPULATION_PASSED = len(_undersized_cells) == 0
print(f"\nmin_tier_population_pct check: {'PASS' if MIN_CELL_POPULATION_PASSED else 'FAIL'} "
      f"(min real cell share required: {_min_cell_pct:.2f}% of HOLDOUT)")
if _undersized_cells:
    for _rl, _tr, _n in _undersized_cells:
        print(f"  UNDERSIZED: {_rl} x {_tr} -- n={_n:,} ({100.0 * _n / _total_holdout:.2f}%)")

ALL_HARD_GATES_PASSED = bool(RISK_LEVEL_MONOTONICITY_PASSED and TREND_COHERENCE_PASSED)
RECOMMENDED_FOR_PRODUCTION = bool(ALL_HARD_GATES_PASSED and MIN_CELL_POPULATION_PASSED)
if not ALL_HARD_GATES_PASSED:
    print(
        "\nHONEST FINDING: one or more hard-gating KPIs did not pass on this real run. This notebook "
        "still proceeds to score and report the full population (Sections 10-14 below) for completeness "
        "-- the same honest 'not yet viable' standard Notebooks 36/40/48 held their own problems to -- but "
        "every downstream artifact and report for Problem 10 marks this run NOT RECOMMENDED FOR PRODUCTION."
    )
print(f"\nRECOMMENDED_FOR_PRODUCTION (this run): {RECOMMENDED_FOR_PRODUCTION}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: FULL CLASSIFICATION METRICS SUITE -- DYNAMIC PD AS A DEFAULT
#             PREDICTOR (HOLDOUT)
# =============================================================================
_section("SECTION 10: Full Classification Metrics Suite -- Dynamic PD as a Default Predictor (Holdout)")

# --- STANDING RULE (2026-08-25, carried from Problems 6/7/8/9): full
#     classification metrics suite, computed and displayed. DYNAMIC_PD is
#     the natural continuous score to evaluate here -- it is the axis the
#     risk-level tiers are built from, and the one real per-customer default
#     probability this notebook produces (STATIC_PD is Problem 1's own,
#     already reported in Problem 1's own metrics). ---
_y_holdout = SCORED_HOLDOUT_DF["target"].to_numpy()
_p_holdout = SCORED_HOLDOUT_DF["DYNAMIC_PD"].to_numpy()

DYNAMIC_PD_ROC_AUC = float(roc_auc_score(_y_holdout, _p_holdout))
DYNAMIC_PD_PR_AUC = float(average_precision_score(_y_holdout, _p_holdout))
DYNAMIC_PD_LOG_LOSS = float(log_loss(_y_holdout, _p_holdout, labels=[0, 1]))


def _metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / len(y_true)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "specificity": float(specificity),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


METRICS_AT_050 = _metrics_at_threshold(_y_holdout, _p_holdout, 0.5)
_precisions, _recalls, _thresholds = precision_recall_curve(_y_holdout, _p_holdout)
_f1s = np.where((_precisions + _recalls) > 0, 2 * _precisions * _recalls / (_precisions + _recalls + 1e-12), 0.0)
_best_idx = int(np.argmax(_f1s[:-1])) if len(_thresholds) > 0 else 0
_f1_optimal_threshold = float(_thresholds[_best_idx]) if len(_thresholds) > 0 else 0.5
METRICS_AT_F1_OPTIMAL = _metrics_at_threshold(_y_holdout, _p_holdout, _f1_optimal_threshold)

print(f"DYNAMIC_PD ROC-AUC (real, HOLDOUT)   : {DYNAMIC_PD_ROC_AUC:.4f}")
print(f"DYNAMIC_PD PR-AUC (real, HOLDOUT)    : {DYNAMIC_PD_PR_AUC:.4f}")
print(f"DYNAMIC_PD Log Loss (real, HOLDOUT)  : {DYNAMIC_PD_LOG_LOSS:.4f}")
print(f"\nMetrics @ 0.5 threshold: {json.dumps(METRICS_AT_050, indent=2)}")
print(f"\nMetrics @ F1-optimal threshold ({_f1_optimal_threshold:.4f}): "
      f"{json.dumps(METRICS_AT_F1_OPTIMAL, indent=2)}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: APPLY THE REAL ACTION-TIER POLICY -- RANKED CREDIT-LINE
#             WORKLIST
# =============================================================================
_section("SECTION 11: Apply the Real Action-Tier Policy -- Ranked Credit-Line Worklist")

_action_map = {(c["risk_level"], c["trend"]): c["action"] for c in ACTION_TIER_MATRIX}
_action_exprs = pl.struct(["RISK_LEVEL", "TREND"]).map_elements(
    lambda s: _action_map[(s["RISK_LEVEL"], s["TREND"])], return_dtype=pl.Utf8
).alias("ACTION")
SCORED_DF = SCORED_DF.with_columns(_action_exprs)

_action_counts = (
    SCORED_DF.group_by("ACTION").agg(pl.len().alias("n"))
    .sort("n", descending=True)
)
print("Real action-tier population counts (full eligible population):")
for _row in _action_counts.iter_rows(named=True):
    print(f"  {_row['ACTION']:<20}: {_row['n']:>8,}  ({100.0 * _row['n'] / SCORED_DF.height:.1f}%)")

CREDIT_LINE_WORKLIST = SCORED_DF.select(
    ["customer_ID", "STATIC_PD", "DYNAMIC_PD", "DYNAMIC_PD_EARLY", "PD_TREND", "RISK_LEVEL", "TREND",
     "ACTION", "target"]
).sort(["ACTION", "DYNAMIC_PD"], descending=[False, True])
print(f"\nReal ranked credit-line worklist built: {CREDIT_LINE_WORKLIST.height:,} customers")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: PERSIST SCORED POPULATION & UPDATED POLICY ARTIFACTS
# =============================================================================
_section("SECTION 12: Persist Scored Population & Updated Policy Artifacts")

worklist_path = CREDIT_LINE_MODELING_DIR / "credit_line_worklist.parquet"
CREDIT_LINE_WORKLIST.write_parquet(worklist_path)
print(f"Wrote: {worklist_path} ({worklist_path.stat().st_size / 1e6:.1f} MB)")

MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 10 -- Credit Line Management (Modeling)",
    "eligible_population": SCORED_DF.height,
    "train_split_population": SCORED_TRAIN_DF.height,
    "holdout_split_population": SCORED_HOLDOUT_DF.height,
    "risk_level_cut_low": RISK_LEVEL_CUT_LOW,
    "risk_level_cut_high": RISK_LEVEL_CUT_HIGH,
    "trend_cut_low": TREND_CUT_LOW,
    "trend_cut_high": TREND_CUT_HIGH,
    "pd_trend_definition": (
        f"DYNAMIC_PD (current, real, most recent {P6_WINNING_W}-statement window) minus "
        f"DYNAMIC_PD_EARLY (same Problem 6 model, real, immediately-preceding non-overlapping "
        f"{P6_WINNING_W}-statement window) -- redefined 2026-08-27 from the original DYNAMIC_PD - "
        "STATIC_PD cross-model residual after discovering it was anti-correlated with real outcomes "
        "(see Notebook 54 Section 6 addendum)."
    ),
    "kpi_results": {
        "risk_level_monotonicity": {
            "strictly_increasing": RISK_LEVEL_MONOTONIC,
            "top_to_bottom_ratio": RISK_LEVEL_RATIO,
            "default_rates_by_tier": _risk_default_rates,
            "passed": RISK_LEVEL_MONOTONICITY_PASSED,
        },
        "trend_coherence": {
            "cells_passed": _trend_coherence_cells_passed,
            "passed": TREND_COHERENCE_PASSED,
        },
        "min_tier_population_pct": {
            "passed": MIN_CELL_POPULATION_PASSED,
            "undersized_cells": [{"risk_level": r, "trend": t, "n": n} for r, t, n in _undersized_cells],
        },
    },
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "dynamic_pd_roc_auc": DYNAMIC_PD_ROC_AUC,
    "dynamic_pd_pr_auc": DYNAMIC_PD_PR_AUC,
    "dynamic_pd_log_loss": DYNAMIC_PD_LOG_LOSS,
    "metrics_at_threshold_0_50": METRICS_AT_050,
    "metrics_at_f1_optimal_threshold": METRICS_AT_F1_OPTIMAL,
    "action_tier_counts": {r["ACTION"]: r["n"] for r in _action_counts.iter_rows(named=True)},
    "worklist_path": str(worklist_path),
    "random_seed": RANDOM_SEED,
}
modeling_results_path = CREDIT_LINE_MODELING_DIR / "credit_line_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 13: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Worklist file was written", worklist_path.exists())
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("STATIC_PD is a real probability in [0, 1] for every scored customer",
                              bool((SCORED_DF["STATIC_PD"] >= 0).all() and (SCORED_DF["STATIC_PD"] <= 1).all()))
_all_checks_passed &= _check("DYNAMIC_PD is a real probability in [0, 1] for every scored customer",
                              bool((SCORED_DF["DYNAMIC_PD"] >= 0).all() and (SCORED_DF["DYNAMIC_PD"] <= 1).all()))
_all_checks_passed &= _check("DYNAMIC_PD_EARLY is a real probability in [0, 1] for every scored customer",
                              bool((SCORED_DF["DYNAMIC_PD_EARLY"] >= 0).all() and (SCORED_DF["DYNAMIC_PD_EARLY"] <= 1).all()))
_all_checks_passed &= _check("Every eligible customer was assigned exactly one RISK_LEVEL",
                              SCORED_DF["RISK_LEVEL"].is_in(RISK_LEVEL_NAMES).all())
_all_checks_passed &= _check("Every eligible customer was assigned exactly one TREND",
                              SCORED_DF["TREND"].is_in(TREND_NAMES).all())
_all_checks_passed &= _check("Every eligible customer was assigned a real action from the 9-cell matrix",
                              SCORED_DF["ACTION"].is_in([c["action"] for c in ACTION_TIER_MATRIX]).all())
_all_checks_passed &= _check("Worklist row count matches the eligible population count",
                              CREDIT_LINE_WORKLIST.height == SCORED_DF.height)
_all_checks_passed &= _check("Risk-level cut values are correctly ordered (low < high)",
                              RISK_LEVEL_CUT_LOW < RISK_LEVEL_CUT_HIGH)
_all_checks_passed &= _check("Trend cut values are correctly ordered (low < high)",
                              TREND_CUT_LOW < TREND_CUT_HIGH)
_all_checks_passed &= _check("DYNAMIC_PD ROC-AUC is a real value in (0.5, 1.0] -- better than random",
                              0.5 < DYNAMIC_PD_ROC_AUC <= 1.0)
_all_checks_passed &= _check("Train and holdout splits do not overlap",
                              len(set(SCORED_TRAIN_DF["customer_ID"]) & set(SCORED_HOLDOUT_DF["customer_ID"])) == 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 13 complete -- all checks passed.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 55 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 14: Write Notebook 55 Summary Artifact")

NB55_SUMMARY = {
    "notebook": "55_credit_line_management_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "worklist_path": str(worklist_path),
    "eligible_population": SCORED_DF.height,
    "risk_level_cut_low": RISK_LEVEL_CUT_LOW,
    "risk_level_cut_high": RISK_LEVEL_CUT_HIGH,
    "trend_cut_low": TREND_CUT_LOW,
    "trend_cut_high": TREND_CUT_HIGH,
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "dynamic_pd_roc_auc": DYNAMIC_PD_ROC_AUC,
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB55_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_55_summary.json"
with open(NB55_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB55_SUMMARY, f, indent=2)
print(f"Wrote: {NB55_SUMMARY_PATH}")

_section("NOTEBOOK 55 COMPLETE")
print(f"Eligible population (real)                    : {SCORED_DF.height:,} customers")
print(f"risk_level_monotonicity                       : {'PASS' if RISK_LEVEL_MONOTONICITY_PASSED else 'FAIL'}")
print(f"trend_coherence                                : {'PASS' if TREND_COHERENCE_PASSED else 'FAIL'}")
print(f"DYNAMIC_PD ROC-AUC (real, HOLDOUT)             : {DYNAMIC_PD_ROC_AUC:.4f}")
print(f"RECOMMENDED_FOR_PRODUCTION (this run)          : {RECOMMENDED_FOR_PRODUCTION}")
print(f"Worklist written to: {worklist_path}")
print(
    "\nNext: 56_credit_line_management_validation_deployment.ipynb -- independently reproduces this "
    "notebook's real pipeline from scratch, bootstraps confidence intervals on both hard-gating KPIs, "
    "verifies the persisted worklist against a fresh reproduction, and packages the real FastAPI limit-"
    "recommendation service."
)
